In [1]:
import torch
import torch.nn as nn
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import numpy as np
import os

try:
    from py_vncorenlp import VnCoreNLP
    rdrsegmenter = VnCoreNLP("https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar", annotators="wseg,pos,ner,parse", max_heap_size='-Xmx2g')
    USE_VNCORENLP = True
except:
    USE_VNCORENLP = False
    import re

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
phobert = AutoModel.from_pretrained("vinai/phobert-base").to(DEVICE)
class PhoBERTSUM(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(
            encoder.config.hidden_size, 1
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        scores = self.classifier(cls_embeddings)
        return scores
model = PhoBERTSUM(phobert).to(DEVICE)
model.eval()   # inference mode
def sent_tokenize(text):
    if not isinstance(text, str):
        return []
    text = text.strip()
    if not text:
        return []
    
    if USE_VNCORENLP:
        try:
            sentences = rdrsegmenter.sent_tokenize(text)
            sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
            return sentences
        except:
            pass
    
    sentences = re.split(r'[.!?]+\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return sentences

def split_sentences(text):
    if not isinstance(text, str):
        return []
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if len(s.strip()) > 10]
def get_top_k_by_ratio(num_sentences, ratio=0.6):
    return max(1, int(num_sentences * ratio))
def encode_sentences(sentences, max_len=256):
    encoded = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )
    return encoded["input_ids"].to(DEVICE), encoded["attention_mask"].to(DEVICE)
@torch.no_grad()
def extractive_summary(text, ratio=0.6):
    sentences = split_sentences(text)
    n = len(sentences)

    if n == 0:
        return ""

    # Nếu chỉ có 1 câu → lấy luôn
    if n == 1:
        return sentences[0]

    top_k = get_top_k_by_ratio(n, ratio)

    input_ids, attention_mask = encode_sentences(sentences)
    scores = model(input_ids, attention_mask)

    scores = scores.squeeze().cpu()

    # ĐẢM BẢO top_idx LUÔN là list
    top_idx = torch.topk(
        scores, min(top_k, n)
    ).indices.tolist()

    if not isinstance(top_idx, list):
        top_idx = [top_idx]

    top_idx.sort()

    summary = " ".join(sentences[i] for i in top_idx)
    return summary

rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def evaluate_summary(candidate, reference):
    if not candidate or not reference:
        return {
            'f1': 0.0,
            'rouge1': 0.0,
            'rougel': 0.0,
            'bertscore': 0.0
        }
    
    rouge_scores = rouge_scorer_obj.score(reference, candidate)
    rouge1 = rouge_scores['rouge1'].fmeasure
    rougel = rouge_scores['rougeL'].fmeasure
    
    try:
        P, R, F1_bert = bert_score([candidate], [reference], lang='vi', verbose=False)
        bertscore = F1_bert.item()
    except:
        bertscore = 0.0
    
    candidate_tokens = set(candidate.lower().split())
    reference_tokens = set(reference.lower().split())
    if len(reference_tokens) == 0:
        f1 = 0.0
    else:
        intersection = candidate_tokens & reference_tokens
        precision = len(intersection) / len(candidate_tokens) if len(candidate_tokens) > 0 else 0
        recall = len(intersection) / len(reference_tokens) if len(reference_tokens) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'f1': f1,
        'rouge1': rouge1,
        'rougel': rougel,
        'bertscore': bertscore
    }

def select_best_summary(summaries_dict):
    if not summaries_dict:
        return None, {}
    
    best_key = None
    best_scores = None
    
    for key, scores in summaries_dict.items():
        if best_key is None:
            best_key = key
            best_scores = scores
            continue
        
        if scores['f1'] > best_scores['f1']:
            best_key = key
            best_scores = scores
        elif scores['f1'] == best_scores['f1']:
            if scores['rougel'] > best_scores['rougel']:
                best_key = key
                best_scores = scores
            elif scores['rougel'] == best_scores['rougel']:
                if scores['rouge1'] > best_scores['rouge1']:
                    best_key = key
                    best_scores = scores
                elif scores['rouge1'] == best_scores['rouge1']:
                    if scores['bertscore'] > best_scores['bertscore']:
                        best_key = key
                        best_scores = scores
    
    return best_key, best_scores




c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [3]:
INPUT_XLSX = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data TX.xlsx"
OUTPUT_XLSX = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_TX_with_summary.xlsx"

df = pd.read_excel(INPUT_XLSX)

if 'f1' not in df.columns:
    df['f1'] = np.nan
if 'rougel' not in df.columns:
    df['rougel'] = np.nan
if 'rouge1' not in df.columns:
    df['rouge1'] = np.nan
if 'bertscore' not in df.columns:
    df['bertscore'] = np.nan

reference_col = None
for col in ['reference_summary', 'gold_summary', 'reference', 'gold']:
    if col in df.columns:
        reference_col = col
        break

for idx, row in df.iterrows():
    content = row["content"]
    if pd.isna(content) or content == "":
        continue
    
    reference = ""
    if reference_col and reference_col in row:
        ref_val = row[reference_col]
        if not pd.isna(ref_val) and ref_val != "":
            reference = str(ref_val)
    elif not pd.isna(row.get("summary", "")) and row.get("summary", "") != "":
        reference = str(row["summary"])
    
    summaries_dict = {}
    
    for retry in range(3):
        summary = extractive_summary(content, ratio=0.6)
        
        if reference and reference.strip():
            scores = evaluate_summary(summary, reference)
        else:
            scores = {'f1': 0.0, 'rouge1': 0.0, 'rougel': 0.0, 'bertscore': 0.0}
        
        summaries_dict[retry] = {
            'summary': summary,
            'f1': scores['f1'],
            'rouge1': scores['rouge1'],
            'rougel': scores['rougel'],
            'bertscore': scores['bertscore']
        }
    
    best_key, best_scores = select_best_summary({
        k: {'f1': v['f1'], 'rouge1': v['rouge1'], 'rougel': v['rougel'], 'bertscore': v['bertscore']}
        for k, v in summaries_dict.items()
    })
    
    if best_key is not None:
        df.at[idx, "summary"] = summaries_dict[best_key]['summary']
        df.at[idx, "f1"] = best_scores['f1']
        df.at[idx, "rougel"] = best_scores['rougel']
        df.at[idx, "rouge1"] = best_scores['rouge1']
        df.at[idx, "bertscore"] = best_scores['bertscore']

df.to_excel(OUTPUT_XLSX, index=False)

print("✅ Đã tạo xong bản tóm tắt trích xuất")


c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Đã tạo xong bản tóm tắt trích xuất


In [1]:
import pandas as pd

data = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_TX_with_summary.xlsx")
data[["f1", "rouge1", "rougel", "bertscore"]].mean()

f1           0.631851
rouge1       0.747049
rougel       0.644642
bertscore    0.853762
dtype: float64

In [ ]:
# Cell setup cho Kaggle - Cài đặt vncorenlp
# Chạy cell này trước khi chạy các cell khác trên Kaggle

!pip install py_vncorenlp

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'

from py_vncorenlp import VnCoreNLP

rdrsegmenter = VnCoreNLP(
    "https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar",
    annotators="wseg,pos,ner,parse",
    max_heap_size='-Xmx2g'
)

print("✅ VnCoreNLP đã được khởi tạo")

In [2]:
import pandas as pd
from difflib import SequenceMatcher
import numpy as np
import re

try:
    from py_vncorenlp import VnCoreNLP
    rdrsegmenter = VnCoreNLP("https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar", annotators="wseg,pos,ner,parse", max_heap_size='-Xmx2g')
    USE_VNCORENLP = True
except:
    USE_VNCORENLP = False

def sent_tokenize(text):
    if not isinstance(text, str):
        return []
    text = text.strip()
    if not text:
        return []
    
    if USE_VNCORENLP:
        try:
            sentences = rdrsegmenter.sent_tokenize(text)
            sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
            return sentences
        except:
            pass
    
    sentences = re.split(r'[.!?]+\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return sentences

def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def label_sentences(content, summary, threshold=0.7):
    if pd.isna(content) or content == "":
        return [], []
    
    if pd.isna(summary) or summary == "":
        content_sentences = sent_tokenize(str(content))
        return content_sentences, [0] * len(content_sentences)
    
    content_sentences = sent_tokenize(str(content))
    summary_sentences = sent_tokenize(str(summary))
    
    labels = []
    summary_set = [s.strip().lower() for s in summary_sentences if s.strip()]
    
    for sent in content_sentences:
        sent_clean = sent.strip()
        if len(sent_clean) < 10:
            labels.append(0)
            continue
        
        sent_lower = sent_clean.lower()
        is_in_summary = False
        
        for sum_sent in summary_set:
            sim = similarity(sent_lower, sum_sent)
            if sim >= threshold:
                is_in_summary = True
                break
        
        labels.append(1 if is_in_summary else 0)
    
    return content_sentences, labels

data = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_TX_with_summary.xlsx")

labeled_data = []

for idx, row in data.iterrows():
    content = row.get("content", "")
    summary = row.get("summary", "")
    
    sentences, labels = label_sentences(content, summary, threshold=0.7)
    
    for sent, label in zip(sentences, labels):
        labeled_data.append({
            'doc_id': idx,
            'sentence': sent,
            'label': label,
            'content': content,
            'summary': summary
        })

df_labeled = pd.DataFrame(labeled_data)

output_path = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset extractive\Data_TX_labeled.xlsx"
df_labeled.to_excel(output_path, index=False)

print(f"✅ Đã gắn nhãn {len(df_labeled)} câu")
print(f"   - Số câu được chọn (label=1): {df_labeled['label'].sum()}")
print(f"   - Số câu không được chọn (label=0): {(df_labeled['label'] == 0).sum()}")
print(f"   - Tỷ lệ: {df_labeled['label'].mean():.2%}")

✅ Đã gắn nhãn 22984 câu
   - Số câu được chọn (label=1): 12704
   - Số câu không được chọn (label=0): 10280
   - Tỷ lệ: 55.27%


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import optuna
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from tqdm import tqdm
from torch.optim import AdamW

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(
    "vinai/phobert-base",
    use_fast=False
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class PhoBERTSUM(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        sent_emb = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(sent_emb)

class SentenceDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data = data.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sentence = str(row['sentence'])
        label = float(row['label'])

        encoding = self.tokenizer(
            sentence,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()

    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate(model, dataloader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs.squeeze(), labels)
            total_loss += loss.item()

            preds = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total if total > 0 else 0
    avg_loss = total_loss / len(dataloader)
    return avg_loss, accuracy

def objective(trial):
    data = pd.read_excel(r"/kaggle/working/dataset-tx/Data_TX_labeled.xlsx")

    train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 5e-4)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])
    max_len = trial.suggest_categorical('max_len', [128, 256])
    num_epochs = 5
    warmup_steps = trial.suggest_int('warmup_steps', 100, 800)

    # Define the tokenizer locally inside the function to avoid NameError
    tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    train_dataset = SentenceDataset(train_data, tokenizer, max_len=max_len)
    val_dataset = SentenceDataset(val_data, tokenizer, max_len=max_len)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    encoder = AutoModel.from_pretrained("vinai/phobert-base").to(DEVICE)
    model = PhoBERTSUM(encoder).to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=learning_rate)
    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    best_val_loss = float('inf')
    patience = 2
    patience_counter = 0

    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
        val_loss, val_acc = validate(model, val_loader, DEVICE)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss

study = optuna.create_study(direction='minimize', pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=10, show_progress_bar=True)

print("Best parameters:")
print(study.best_params)
print(f"Best validation loss: {study.best_value:.4f}")

best_params = study.best_params

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
import pandas as pd
import os
from tqdm import tqdm
from torch.optim import AdamW

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(
    "vinai/phobert-base",
    use_fast=False
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class PhoBERTSUM(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Linear(encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        sent_emb = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(sent_emb)

class SentenceDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data = data.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        sentence = str(row['sentence'])
        label = float(row['label'])

        encoding = self.tokenizer(
            sentence,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    for batch in tqdm(dataloader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def validate(model, dataloader, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs.squeeze(), labels)
            total_loss += loss.item()
            
            preds = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    accuracy = correct / total if total > 0 else 0
    avg_loss = total_loss / len(dataloader)
    return avg_loss, accuracy

data = pd.read_excel(r"/kaggle/working/dataset-tx/Data_TX_labeled.xlsx")

train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
test_data, val_data = train_test_split(val_data, test_size=0.5, random_state=42)

best_params= {'learning_rate': 3.868107453864181e-05, 'batch_size': 32, 'max_len': 128, 'warmup_steps': 213}

learning_rate = best_params['learning_rate']
batch_size = best_params['batch_size']
max_len = best_params['max_len']
num_epochs = 10
warmup_steps = best_params['warmup_steps']

train_dataset = SentenceDataset(train_data, tokenizer, max_len=max_len)
val_dataset = SentenceDataset(val_data, tokenizer, max_len=max_len)
test_dataset = SentenceDataset(test_data, tokenizer, max_len=max_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

encoder = AutoModel.from_pretrained("vinai/phobert-base").to(DEVICE)
model = PhoBERTSUM(encoder).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

best_val_loss = float('inf')
best_model_state = None
patience = 3
patience_counter = 0

print(f"Training với tham số tối ưu:")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Max length: {max_len}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Epochs: {num_epochs}")
print()

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
    val_loss, val_acc = validate(model, val_loader, DEVICE)
    
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"  ✓ Model improved!")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{patience})")
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break
    
    print()

if best_model_state:
    model.load_state_dict(best_model_state)

test_loss, test_acc = validate(model, test_loader, DEVICE)
print(f"\nFinal Test Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")

model_save_path = r"/kaggle/working/phobert_summary_model.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'best_params': best_params,
    'test_loss': test_loss,
    'test_acc': test_acc
}, model_save_path)

print(f"\n✅ Model đã được lưu tại: {model_save_path}")